# Language Identification - End-to-End Notebook

This notebook walks through the full pipeline: dataset loading, EDA, preprocessing, feature engineering, model comparison and final prediction. All logic lives in the `src` package.

## 1. Setup and dataset

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.utils import setup_logging
setup_logging()

from src.data_loader import load_dataset
df = load_dataset()
df.head()

In [ ]:
print('Rows:', len(df))
print('Languages:', df['language'].nunique())
df['language'].value_counts()

## 2. Exploratory data analysis

In [ ]:
from src.eda import run_eda
from src.config import REPORTS_DIR
run_eda(df, REPORTS_DIR)

In [ ]:
from IPython.display import Image, display
for f in ['language_frequency_bar.png','text_length_histogram.png','word_count_analysis.png','text_length_boxplot.png']:
    display(Image(f'../reports/{f}'))

## 3. Preprocessing

In [ ]:
from src.preprocessing import clean_text, clean_series
samples = [
  'Hello  world!!!  https://example.com test@mail.com 😀',
  '  Bonjour\tle monde   ',
  'नमस्ते आप कैसे हैं',
]
for s in samples:
    print(repr(s), '->', repr(clean_text(s)))

## 4. Feature engineering comparison

In [ ]:
from src.feature_engineering import get_vectorizer, supported_strategies
print('Strategies:', supported_strategies())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from src.preprocessing import clean_series
from sklearn.preprocessing import LabelEncoder
import numpy as np

X = clean_series(df['text']).to_numpy()
le = LabelEncoder()
y = le.fit_transform(df['language'])

rng = np.random.RandomState(42)
idx = rng.choice(len(X), size=8000, replace=False)
Xs, ys = X[idx], y[idx]
results = {}
for strat in supported_strategies():
    vec = get_vectorizer(strat, max_features=30000)
    M = vec.fit_transform(Xs)
    scores = cross_val_score(LogisticRegression(max_iter=1000), M, ys, cv=3, n_jobs=-1)
    results[strat] = round(scores.mean(), 4)
    print(f'{strat:16} {results[strat]:.4f}')
pd.Series(results).sort_values(ascending=False)

## 5. Training pipeline

Run the full pipeline (`select_best_feature_strategy` -> `compare_models` -> final fit). This can take a few minutes.

In [ ]:
# Full pipeline - uncomment to run
# from src.train import train_and_evaluate
# meta = train_and_evaluate()
# print(meta)

## 6. Prediction demo

In [ ]:
from src.predict import get_predictor
predictor = get_predictor()

for text in ["Hello, how are you?", "Bonjour tout le monde", "नमस्ते आप कैसे हैं", "வணக்கம்", "ഹലോ സുഖമാണോ"]:
    r = predictor.predict(text)
    print(f'{text!r:35} -> {r["language"]:12} {r["confidence"]*100:5.1f}%  ({r["prediction_time_ms"]:.1f} ms)')

## 7. Evaluation metrics

Metrics for the deployed model are stored in `reports/final_metrics.json` and `reports/model_comparison.csv`.

In [ ]:
import json, pandas as pd
from src.config import REPORTS_DIR
m = json.loads((REPORTS_DIR/'final_metrics.json').read_text())
print('Accuracy:', m['accuracy'])
print('F1 macro:', m['f1_macro'])
print('ROC AUC macro:', m.get('roc_auc_macro'))
pd.read_csv(REPORTS_DIR/'model_comparison.csv', index_col=0)